# Call Forex Trading Agent from Fabric

This notebook invokes **Microsoft Agent Framework 1.0** agents using the **Foundry Agent Service v2** with the **OpenAI Responses API**.

The forex trading agents are provisioned in Azure AI Foundry and provide:
- Real-time FX quotes (bid/ask/spread)
- Market status (trend, volatility, day statistics)  
- Trading account management
- Trade execution (buy/sell)
- Position management

**Available Agents**:
- `fx-agent-chatbot` — General-purpose trading assistant
- `fx-agent-research` — Market research analyst
- `fx-agent-suggestion` — Trade suggestion advisor
- `fx-agent-trader` — Trade execution agent

In [ ]:
%pip install azure-ai-projects azure-identity

## Configuration

In [ ]:
# Format: "https://resource_name.services.ai.azure.com/api/projects/project_name"
PROJECT_ENDPOINT = "https://fxag-foundry.services.ai.azure.com/api/projects/fxag-foundry-project"
AGENT_NAME = "fx-agent-chatbot"

## Initialize Client

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Create project client
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Get OpenAI client for Responses API
openai = project.get_openai_client()

print(f"Connected to project: {PROJECT_ENDPOINT}")
print(f"Agent: {AGENT_NAME}")

## Simple Query - Get Current Quote

In [ ]:
# Non-streaming response
response = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "type": "agent_reference",
        }
    },
    input="What is the current AUD/USD quote?",
)

print(response.output_text)

## Streaming Response - Market Status

In [ ]:
# Streaming response for real-time output
stream = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "type": "agent_reference",
        }
    },
    input="What is the current market status?",
    stream=True,
)

print("Agent Response:")
print("=" * 60)
for event in stream:
    if hasattr(event, "delta") and event.delta:
        print(event.delta, end="", flush=True)
print("\n" + "=" * 60)

## Multi-Turn Conversation

In [ ]:
# Create a conversation for multi-turn chat
conversation = openai.conversations.create()
print(f"Conversation ID: {conversation.id}")

# First turn
response = openai.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="Show me my trading accounts",
)
print("\n=== Turn 1 ===")
print(response.output_text)

# Follow-up question in the same conversation
response = openai.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="What is the balance of the first account?",
)
print("\n=== Turn 2 ===")
print(response.output_text)

## View Tool Calls in Response

In [ ]:
# Generate a response and inspect tool calls
response = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "type": "agent_reference",
        }
    },
    input="Get the current quote and market status for AUD/USD",
)

print("=== Response Output ===")
for item in response.output:
    if item.type == "tool_call":
        print(f"\n[Tool Call: {item.function.name}]")
        print(f"Arguments: {item.function.arguments}")
    elif item.type == "tool_call_result":
        print(f"[Tool Result: {item.tool_call_id}]")
    elif item.type == "message":
        print(f"\n{item.content}")

## Switch to Different Agent

In [ ]:
# Use the research analyst agent
research_response = openai.responses.create(
    extra_body={
        "agent_reference": {
            "name": "fx-agent-research",
            "type": "agent_reference",
        }
    },
    input="Analyze today's price history and identify key patterns",
    stream=True,
)

print("Research Agent:")
print("=" * 60)
for event in research_response:
    if hasattr(event, "delta") and event.delta:
        print(event.delta, end="", flush=True)
print("\n" + "=" * 60)

## Sample Questions to Try

**With `fx-agent-chatbot`** (general assistant):
```python
openai.responses.create(
    extra_body={"agent_reference": {"name": "fx-agent-chatbot", "type": "agent_reference"}},
    input="Show me today's price history"
)
```

**With `fx-agent-research`** (market analyst):
```python
openai.responses.create(
    extra_body={"agent_reference": {"name": "fx-agent-research", "type": "agent_reference"}},
    input="Analyze the current market conditions and identify trends"
)
```

**With `fx-agent-suggestion`** (trade advisor):
```python
# Create a conversation first
conv = openai.conversations.create()
openai.responses.create(
    conversation=conv.id,
    extra_body={"agent_reference": {"name": "fx-agent-suggestion", "type": "agent_reference"}},
    input="Review my portfolio and suggest potential trades"
)
```

**With `fx-agent-trader`** (execution agent):
```python
openai.responses.create(
    extra_body={"agent_reference": {"name": "fx-agent-trader", "type": "agent_reference"}},
    input="Execute a buy order for 1 lot on account ACC001"
)
```